# Tensorflow2.0 小练习

In [1]:
import tensorflow as tf
import numpy as np

I0000 00:00:1774189694.175611  446790 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774189694.208506  446790 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1774189695.134460  446790 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 实现softmax函数

In [2]:
def softmax(x):
    # 对最后一维进行softmax归一化
    # 先减去最大值，防止指数溢出（数值稳定性）
    x = tf.cast(x, tf.float64)
    x_max = tf.reduce_max(x, axis=-1, keepdims=True)
    x_exp = tf.exp(x - x_max)
    partition = tf.reduce_sum(x_exp, axis=-1, keepdims=True)
    prob_x = x_exp / partition
    return prob_x

test_data = np.random.normal(size=[10, 5])
(softmax(test_data).numpy() - tf.nn.softmax(test_data, axis=-1).numpy())**2 <0.0001

W0000 00:00:1774189698.539064  446790 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


array([[ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True]])

## 实现sigmoid函数

In [3]:
def sigmoid(x):
    # sigmoid(x) = 1 / (1 + exp(-x))
    x = tf.cast(x, tf.float64)
    prob_x = 1.0 / (1.0 + tf.exp(-x))
    return prob_x

test_data = np.random.normal(size=[10, 5])
(sigmoid(test_data).numpy() - tf.nn.sigmoid(test_data).numpy())**2 < 0.0001

array([[ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True]])

## 实现 softmax 交叉熵loss函数

In [4]:
def softmax_ce(x, label):
    # x 是 softmax 之后的概率值，label 是 one-hot 标签
    # 交叉熵 = -sum(label * log(prob)) 然后取均值
    epsilon = 1e-12  # 防止log(0)
    loss = tf.reduce_mean(-tf.reduce_sum(label * tf.math.log(x + epsilon), axis=-1))
    return loss

test_data = np.random.normal(size=[10, 5])
prob = tf.nn.softmax(test_data)
label = np.zeros_like(test_data)
label[np.arange(10), np.random.randint(0, 5, size=10)]=1.

((tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(label, test_data))
  - softmax_ce(prob, label))**2 < 0.0001).numpy()

np.True_

## 实现 sigmoid 交叉熵loss函数

In [5]:
def sigmoid_ce(x, label):
    # x 是 sigmoid 之后的概率值，label 是二分类标签
    # 二元交叉熵 = -[label * log(prob) + (1-label) * log(1-prob)]
    epsilon = 1e-12  # 防止log(0)
    loss = tf.reduce_mean(-(label * tf.math.log(x + epsilon) + (1 - label) * tf.math.log(1 - x + epsilon)))
    return loss

test_data = np.random.normal(size=[10])
prob = tf.nn.sigmoid(test_data)
label = np.random.randint(0, 2, 10).astype(test_data.dtype)
print (label)

((tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(label, test_data))- sigmoid_ce(prob, label))**2 < 0.0001).numpy()

[1. 0. 0. 1. 1. 0. 0. 0. 0. 1.]


np.True_